# SmartPantry AI | Master Integration & Verification Notebook

**Student capstone demonstration | Mocked external services | GitHub-ready**

This notebook runs the **real project modules** and their existing automated tests, while replacing external API and database responses with deterministic mocks. It does not call OpenAI, Spoonacular, Open Food Facts, FDA/USDA endpoints, or Supabase during the demonstrations.

**How to run:** Open this notebook from the repository in Google Colab (or Jupyter), then select **Runtime → Run all**. Internet access is needed only to clone the repository (if not already available) and install Python dependencies. The test scenarios use simulated data and are **not evidence of a live recall check, actual food safety, or production database security**.

**Repository:** https://github.com/SmartPantry-AI/smartpantry-ai

**Source basis:** The modules and tests in the supplied `smartpantry-ai-main` repository snapshot. No application modules are modified by this notebook.

## 1. Setup and reproducibility
The cell detects a checked-out repository first; in Colab it clones `main` if needed. The dependency installation uses the repository’s own `requirements.txt`.

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SmartPantry-AI/smartpantry-ai.git"

def find_repo():
    for candidate in [Path.cwd(), *Path.cwd().parents, Path.cwd()/"smartpantry-ai", Path.cwd()/"smartpantry-ai-main", Path("/content/smartpantry-ai")]:
        if (candidate / "modules" / "ai_assistant.py").is_file() and (candidate / "requirements.txt").is_file():
            return candidate.resolve()
    target = Path.cwd() / "smartpantry-ai"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    return target.resolve()

ROOT = find_repo()
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
print("Dependencies installed; repository ready.")

Repository: /content/smartpantry-ai
Python: 3.13.18
Dependencies installed; repository ready.


## 2. Run the existing project test suite
This executes the project’s original tests unchanged. Any failure remains visible and causes the notebook to stop, rather than displaying a misleading success summary. Existing test coverage is separate from the integration scenario below.

The existing Mochi tests patch the OpenAI client but first require a nonempty `OPENAI_API_KEY`. We provide a **dummy placeholder** before pytest runs. It is not a real credential and must never be used for live requests.


In [2]:
import pytest

# Placeholder only: production OpenAI calls are mocked by the existing tests.
# Mochi checks for a non-empty key before invoking its patched OpenAI client.
os.environ["OPENAI_API_KEY"] = "mock-not-a-real-api-key"
TEST_FILES = sorted((ROOT / "tests").glob("test_*.py"))
assert TEST_FILES, "No project test files were found"
print(f"Found {len(TEST_FILES)} project test files")
exit_code = pytest.main([str(ROOT / "tests"), "-q", "--disable-warnings", "--tb=short", "-p", "no:cacheprovider"])
assert exit_code == 0, f"Project test suite did not pass (exit code {exit_code})"
print("PASS: existing project test suite")

Found 11 project test files
........................................................................ [ 45%]
........................................................................ [ 90%]
................                                                         [100%]
160 passed in 4.16s
PASS: existing project test suite


## 3. Component demonstrations
The examples below exercise the imported production functions. Only the **external dependencies** (network responses, AI model responses, and database client) are simulated.

In [3]:
from unittest.mock import MagicMock, patch
from tempfile import TemporaryDirectory
from modules import barcode_lookup, image_recognition, storage_guidance, recall_checker, recipe_recommender, ai_assistant, supabase_pantry
from modules.pantry_manager import PantryManager

checks = {}
def verify(name, condition, details=None):
    assert condition, f"FAIL: {name} | {details}"
    checks[name] = "PASS"
    print(f"PASS: {name}" + (f" | {details}" if details is not None else ""))


### 3A. Barcode → Open Food Facts product record (mocked lookup)
We mock the barcode detector result and the HTTP response; the product-record transformation remains real.

In [4]:
barcode_payload = {"product": {"product_name": "Demo Oats", "brands": "Example Brand", "categories": "Breakfast cereals", "quantity": "18 oz"}}
mock_http = MagicMock()
mock_http.json.return_value = barcode_payload
with patch.object(barcode_lookup, "decode_barcode", return_value=({"barcode": "041220892617", "barcode_type": "UPC_A"}, None)), \
     patch.object(barcode_lookup.requests, "get", return_value=mock_http) as get_request:
    barcode_record, barcode_error = barcode_lookup.identify_product("mocked_barcode.jpg")
verify("Barcode product normalization", barcode_error is None and barcode_record["product_name"] == "Demo Oats" and barcode_record["user_confirmed"] is False, barcode_record)
get_request.assert_called_once()


PASS: Barcode product normalization | {'barcode': '041220892617', 'barcode_type': 'UPC_A', 'product_name': 'Demo Oats', 'brand': 'Example Brand', 'source_category': 'Breakfast cereals', 'quantity': '18 oz', 'product_data_source': 'Open Food Facts', 'input_method': 'barcode_image', 'user_confirmed': False}


### 3B. Food image recognition (mocked OpenAI response)
The real image-reading and response-extraction function executes, but no image is sent to OpenAI.

In [5]:
with TemporaryDirectory() as folder:
    sample_image = Path(folder) / "food.jpg"
    sample_image.write_bytes(b"synthetic test image bytes")
    fake_client = MagicMock()
    fake_client.responses.create.return_value.output_text = "Tomato"
    with patch.object(image_recognition.os, "getenv", return_value="mock-test-key"), \
         patch.object(image_recognition, "OpenAI", return_value=fake_client):
        recognized = image_recognition.recognize_food(sample_image)
verify("Image recognition response handling", recognized == "Tomato", recognized)


PASS: Image recognition response handling | Tomato


### 3C. Storage rules (real repository CSVs)
These datasets are loaded from the repository, not mocked. This verifies that the expected files and column formats are present.

In [6]:
storage_rules = storage_guidance.load_storage_rules()
foodkeeper_rules = storage_guidance.load_foodkeeper_rules()
osu_rules = storage_guidance.load_osu_rules()
verify("Storage datasets load", all(len(x) > 0 for x in (storage_rules, foodkeeper_rules, osu_rules)), f"FoodSafety.gov: {len(storage_rules)}, FoodKeeper: {len(foodkeeper_rules)}, OSU: {len(osu_rules)}")


PASS: Storage datasets load | FoodSafety.gov: 20, FoodKeeper: 17, OSU: 15


### 3D. Recall matching and safety gate (synthetic recall)
A fabricated recall is used **only for testing**. This does not check whether a real product is recalled.

In [7]:
synthetic_product = {"product_name": "Demo Oats", "brand": "Example Brand", "upc": "041220892617", "lot_code": "DEMO-LOT"}
synthetic_recall = {"source": "FDA", "recall_id": "DEMO-ONLY", "product_description": "Demo Oats", "brand": "Example Brand", "upc": "041220892617", "lot_codes": ["DEMO-LOT"], "recall_status": "Ongoing", "reason_for_recall": "Synthetic test scenario", "source_url": "https://example.invalid/demo"}
recall_match = recall_checker.match_recall_to_product(synthetic_product, synthetic_recall)
verify("Recall match flags potential issue", recall_match["status"] == "POSSIBLE_MATCH", recall_match)


PASS: Recall match flags potential issue | {'status': 'POSSIBLE_MATCH'}


### 3E. Recipe API normalization (mocked Spoonacular response)
The real fetch function and conversion logic are used; the HTTP response is synthetic.

In [8]:
synthetic_recipes = [{"title": "Tomato and Spinach Bowl", "usedIngredients": [{"name": "tomato"}, {"name": "spinach"}], "missedIngredients": [{"name": "rice"}]}]
recipe_http = MagicMock()
recipe_http.json.return_value = synthetic_recipes
with patch.object(recipe_recommender.requests, "get", return_value=recipe_http) as recipe_get:
    raw_recipes = recipe_recommender.fetch_spoonacular_recipes(["tomato", "spinach"])
normalized_recipe = recipe_recommender.convert_spoonacular_recipe(raw_recipes[0])
verify("Recipe fetch and normalization", normalized_recipe["name"] == "Tomato and Spinach Bowl" and "rice" in normalized_recipe["ingredients"], normalized_recipe)
recipe_get.assert_called_once()


PASS: Recipe fetch and normalization | {'name': 'Tomato and Spinach Bowl', 'ingredients': ['tomato', 'spinach', 'rice']}


### 3F. Mochi assistant (mocked OpenAI response)
The production assistant function is called, while the external model response is controlled.

In [9]:
fake_mochi = MagicMock()
fake_mochi.responses.create.return_value.output_text = "I can help organize your pantry!"
with patch.object(ai_assistant.os, "getenv", return_value="mock-test-key"), \
     patch.object(ai_assistant, "OpenAI", return_value=fake_mochi):
    mochi_reply = ai_assistant.ask_ai("What can you help with?")
verify("Mochi response handling", "organize your pantry" in mochi_reply, mochi_reply)


PASS: Mochi response handling | I can help organize your pantry!


### 3G. Supabase pantry operations (mock client)
The real `add_pantry_item` and `load_user_pantry` functions are used. No real accounts or rows are created. Database row-level security cannot be verified using a mock.

In [10]:
db = MagicMock()
db.auth.get_user.return_value.user.id = "demo-user"
db.table.return_value.insert.return_value.execute.return_value.data = [{"id": 1, "user_id": "demo-user", "name": "Tomato", "quantity": 2, "unit": "each"}]
inserted = supabase_pantry.add_pantry_item(db, "Tomato", 2, "each")
insert_payload = db.table.return_value.insert.call_args.args[0]
verify("Supabase authenticated insert payload", inserted[0]["name"] == "Tomato" and insert_payload["user_id"] == "demo-user", insert_payload)

db_read = MagicMock()
db_read.table.return_value.select.return_value.order.return_value.execute.return_value.data = inserted
loaded = supabase_pantry.load_user_pantry(db_read)
verify("Supabase pantry read", loaded == inserted, loaded)


PASS: Supabase authenticated insert payload | {'user_id': 'demo-user', 'name': 'Tomato', 'quantity': 2, 'unit': 'each', 'purchase_date': None, 'expiration_date': None, 'storage_location': None, 'prepared_date': None}
PASS: Supabase pantry read | [{'id': 1, 'user_id': 'demo-user', 'name': 'Tomato', 'quantity': 2, 'unit': 'each'}]


## 4. End-to-end simulated pantry scenario
This ties the demonstrated functions together: **identify product → flag possible recall → hold for review**, and separately **identify a food image → add confirmed food → read pantry → get a recipe → ask Mochi**. The scenario does not imply a production UI flow or a verified live recall result.

In [11]:
# Scenario A: potentially recalled product is not added automatically.
blocked_product = barcode_record
potential_recall = recall_checker.match_recall_to_product(
    {"product_name": blocked_product["product_name"], "upc": blocked_product["barcode"]},
    synthetic_recall,
)
verify("Safety review before auto-add", potential_recall["status"] == "POSSIBLE_MATCH" and blocked_product["user_confirmed"] is False)

# Scenario B: a user confirms the separate, mock-recognized food.
local_pantry = PantryManager()
confirmed_item = local_pantry.add_item(recognized, 2, "each")
verify("Confirmed food added to in-memory pantry", confirmed_item["name"] == "tomatoes" and len(local_pantry.get_items()) == 1, local_pantry.get_items())

# Simulated persistence of this confirmed item using the actual Supabase module.
scenario_db = MagicMock()
scenario_db.auth.get_user.return_value.user.id = "demo-user"
scenario_db.table.return_value.insert.return_value.execute.return_value.data = [{"id": 2, "user_id": "demo-user", "name": confirmed_item["name"], "quantity": 2, "unit": "each"}]
saved = supabase_pantry.add_pantry_item(scenario_db, confirmed_item["name"], confirmed_item["quantity"], confirmed_item["unit"])
verify("Confirmed item sent to mocked persistence layer", saved[0]["name"] == "tomatoes")
verify("Recipe available for display", bool(normalized_recipe["name"]))
verify("Mochi available for display", bool(mochi_reply))
print("\nScenario summary: synthetic recalled oats held for review; confirmed tomato processed with mocked persistence; mocked recipe and Mochi outputs available.")


PASS: Safety review before auto-add
PASS: Confirmed food added to in-memory pantry | [{'name': 'tomatoes', 'quantity': 2, 'unit': 'each'}]
PASS: Confirmed item sent to mocked persistence layer
PASS: Recipe available for display
PASS: Mochi available for display

Scenario summary: synthetic recalled oats held for review; confirmed tomato processed with mocked persistence; mocked recipe and Mochi outputs available.


## 5. Final results and limitations
Every check must pass for this notebook to finish. These demonstrations complement the existing pytest suite; they do not replace live acceptance testing or validate real-time recall coverage, model accuracy, production authentication, or full Streamlit UI behavior.

In [12]:
print("SMARTPANTRY MASTER NOTEBOOK RESULTS")
print("=" * 44)
for name, status in checks.items():
    print(f"{status:4} | {name}")
print("=" * 44)
print(f"{len(checks)}/{len(checks)} demonstration checks passed")
assert checks and all(value == "PASS" for value in checks.values())
print("Existing pytest suite: PASSED (Section 2)")
print("External service requests in demonstration cells: MOCKED")
print("Master notebook completed successfully.")

SMARTPANTRY MASTER NOTEBOOK RESULTS
PASS | Barcode product normalization
PASS | Image recognition response handling
PASS | Storage datasets load
PASS | Recall match flags potential issue
PASS | Recipe fetch and normalization
PASS | Mochi response handling
PASS | Supabase authenticated insert payload
PASS | Supabase pantry read
PASS | Safety review before auto-add
PASS | Confirmed food added to in-memory pantry
PASS | Confirmed item sent to mocked persistence layer
PASS | Recipe available for display
PASS | Mochi available for display
13/13 demonstration checks passed
Existing pytest suite: PASSED (Section 2)
External service requests in demonstration cells: MOCKED
Master notebook completed successfully.


### Interpretation

- **Real code exercised:** existing SmartPantry module functions, repository datasets, and existing automated tests.
- **Mocked:** OpenAI, Open Food Facts, Spoonacular, Supabase, and synthetic recall data. The demonstration does not make live API calls.
- **Not established here:** actual food identification accuracy, real product recall status, true end-to-end Streamlit interaction, live database isolation/RLS, and real external API availability.
- **Safety:** A potential recall match is a reason for review, not proof of a recall. Do not use simulated data to make food safety decisions.

**Professor:** Run all cells in order. A failed assertion or pytest run means the notebook has **not** passed. The final printed summary appears only after all earlier cells succeed.